# Paper figures: public mean-10 results

This notebook reads the public mean-10 metrics for all four training
quantities and source OOD labels 2–5. It creates grouped error bars, building
heatmaps, architecture tables, and the fixed-gate sensitivity curves used for
paper analysis.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

CODE_DIR = Path.cwd()
if not (CODE_DIR / 'config.py').is_file():
    raise FileNotFoundError('Run this notebook with post_processing as the working directory.')
sys.path.insert(0, str(CODE_DIR))
import config as cfg
FINAL_ROOT = cfg.FINAL_ROOT
RUN_ROOT = None  # Set only if saved public outputs remain in another Result/OOD subfolder.
PROMPT_STYLE = 'public'  # public | original；仅切换读取位置
SAMPLES = 10
TRAIN_DAYS = ['14-day', '21-day', '28-day', 'Full']
OODS = ['OOD-2', 'OOD-3', 'OOD-4', 'OOD-5']
TARGET_MATCHING = 'legacy'
BUILDINGS = []  # 空列表读取下列7栋；也可填建筑全名，只画选中的建筑

if PROMPT_STYLE not in ('public','original'):
    raise ValueError('PROMPT_STYLE must be public or original')
S = cfg.settings(final_root=FINAL_ROOT, run_root=RUN_ROOT, samples=SAMPLES, prompt_style=PROMPT_STYLE,
    train_days=TRAIN_DAYS, oods=OODS, buildings=BUILDINGS,
    prior_source='generated', target_matching=TARGET_MATCHING)
RUN = cfg.metric_root(S)
OUT = RUN / 'paper_figures_36_all_training_days'
KEYS = ['Architecture','Method','Model_Tag','Seed','Building','Train_Days','OOD_Label','External_Variant']
GROUP = ['Building','Train_Days','OOD_Label']
METRICS = ['RMSE','MAE']
ALPHAS = [round(i/20,2) for i in range(21)]
RULES = ['Rule-%.2f'%a for a in ALPHAS]
MAIN_RULES = ['Rule-%.2f'%a for a in [0,.2,.4,.6,.8,1]]
OURS = 'LLM event+phase'
MAIN = MAIN_RULES + [OURS]
COLORS = ['#cccccc','#b9cbd7','#91abc0','#718ca8','#586c8a','#3d4f68','#dc863d']
NAMES = ['Raw','Fixed 0.2','Fixed 0.4','Fixed 0.6','Fixed 0.8','Fixed 1.0','Ours']
B_NAMES = {'10zone_RetailStripmall_C_1':'Retail','118zone_Healthcare_1':'Healthcare',
 '18zone_OfficeMedium_B_1':'Office','22zone_HotelLarge_B_1':'Hotel Large',
 '25zone_SchoolPrimary_1':'Primary School','46zone_SchoolSecondary_1':'Secondary School',
 '67zone_HotelSmall_1':'Hotel Small'}
SELECTED_BUILDINGS = BUILDINGS or list(B_NAMES)
plt.rcParams.update({'font.size':14,'pdf.fonttype':42,'axes.spines.top':False,'axes.spines.right':False})
def save(fig,name):
    if not coverage.Complete.all():
        fig.suptitle('Available paired cases (partial coverage)', fontsize=13)
    for ext in ('pdf','png'):
        fig.savefig(OUT/(name+'.'+ext),bbox_inches='tight',dpi=200)
    plt.show()
    plt.close(fig)

source_file = RUN/'case_metrics.csv'
columns = KEYS + ['Postprocess_Method','MAE','RMSE','Points']
if source_file.is_file():
    try:
        CASES = pd.read_csv(source_file, keep_default_na=False, dtype={k:str for k in KEYS})
    except pd.errors.EmptyDataError:
        CASES = pd.DataFrame(columns=columns)
else:
    print('本组指标尚未写出：', source_file)
    print('当前不会改读其他samples或prompt；稍后重新运行即可。')
    CASES = pd.DataFrame(columns=columns)
CASES = CASES.loc[CASES.Building.isin(SELECTED_BUILDINGS) & CASES.Train_Days.isin(TRAIN_DAYS)
    & CASES.OOD_Label.isin(OODS) & CASES.Postprocess_Method.isin(RULES+[OURS])].copy()
if CASES.empty:
    print('当前选择尚无已保存指标，先显示计划覆盖，暂不画图。')
if CASES.duplicated(KEYS+['Postprocess_Method']).any():
    raise ValueError('同case/strategy存在重复指标')
for column in METRICS+['Points']:
    CASES[column] = pd.to_numeric(CASES[column],errors='coerce')
CASES['_finite'] = np.isfinite(CASES[METRICS+['Points']].to_numpy(float)).all(axis=1)
state = CASES.groupby(KEYS).agg(Strategies=('Postprocess_Method','nunique'),
    Same_Points=('Points','nunique'),Finite=('_finite','all'),Points=('Points','min'))
valid = state.Strategies.eq(len(RULES)+1) & state.Same_Points.eq(1) & state.Finite & state.Points.gt(0)
complete = state.loc[valid].reset_index()[KEYS]
print('选中的case数：',len(state),'；策略齐全且Points一致：',len(complete),'；排除：',int((~valid).sum()))
CASES = CASES.merge(complete,on=KEYS,how='inner',validate='many_to_one').drop(columns='_finite')
if CASES.empty:
    print('尚无21个固定gate与Ours全部齐全的同case结果；请等任务继续写出结果。')

PLAN = pd.read_csv(FINAL_ROOT/'OOD/postprocessing/case_manifest.csv',keep_default_na=False,dtype={k:str for k in KEYS})
PLAN = PLAN.loc[PLAN.Building.isin(SELECTED_BUILDINGS) & PLAN.Train_Days.isin(TRAIN_DAYS)
    & PLAN.OOD_Label.isin(OODS),KEYS].drop_duplicates()
if len(complete.merge(PLAN,on=KEYS)) != len(complete):
    raise ValueError('已有指标case与当前case索引不一致')
coverage = PLAN.groupby(GROUP).size().rename('Planned_Cases').reset_index().merge(
    complete.groupby(GROUP).size().rename('Paired_Cases').reset_index(),on=GROUP,how='left')
coverage['Paired_Cases'] = coverage.Paired_Cases.fillna(0).astype(int)
coverage['Complete'] = coverage.Planned_Cases.eq(coverage.Paired_Cases)
display(coverage)
OUT.mkdir(parents=True,exist_ok=True)
coverage.to_csv(OUT/'coverage.csv',index=False)
if not coverage.Complete.all():
    print('以下图表仅表示当前可用配对子集，覆盖不完整。')
DAYS = [d for d in TRAIN_DAYS if d in set(CASES.Train_Days)]
OODS = [o for o in OODS if o in set(CASES.OOD_Label)]
BUILD = CASES.groupby(GROUP+['Postprocess_Method'],as_index=False)[METRICS].mean()
OVER = BUILD.groupby(['Train_Days','OOD_Label','Postprocess_Method'],as_index=False)[METRICS].mean()
AB = CASES.groupby(['Architecture']+GROUP+['Postprocess_Method'],as_index=False)[METRICS].mean()
ARCH = AB.groupby(['Architecture','Train_Days','OOD_Label','Postprocess_Method'],as_index=False)[METRICS].mean()
BUILD.to_csv(OUT/'building_metrics.csv',index=False)
OVER.to_csv(OUT/'overall_building_equal.csv',index=False)
print('读取：',source_file)
print('Prompt：',PROMPT_STYLE,'；samples：',SAMPLES,'；配对case：',len(complete),'；训练规模：',DAYS,'；OOD：',OODS)
print('图表与CSV输出：',OUT)
print('完整覆盖：',bool(coverage.Complete.all()),'；已完成：',len(complete),'/',len(PLAN))


## 1. Main grouped bars: training days on the x axis

沿用36版布局，每个OOD一幅面板，比较六种预设固定gate与Ours。当前只有Full，x轴只画Full，不补出其他训练规模。


In [ ]:
if CASES.empty:
    print('本组暂时没有完整配对指标，跳过此类图；稍后重新运行。')
else:
    for metric in METRICS:
        oods=[o for o in OODS if o in set(OVER.OOD_Label)]
        fig,axes=plt.subplots(1,len(oods),figsize=(7*len(oods),5.6),squeeze=False,constrained_layout=True)
        for ax,ood in zip(axes[0],oods):
            tab=OVER.loc[OVER.OOD_Label.eq(ood)].pivot(index='Train_Days',columns='Postprocess_Method',values=metric)
            days=[d for d in DAYS if d in tab.index];tab=tab.reindex(days)[MAIN]
            x=np.arange(len(days));w=.11
            for j,(method,color,label) in enumerate(zip(MAIN,COLORS,NAMES)):
                bars=ax.bar(x+(j-3)*w,tab[method],w,color=color,label=label)
                if method==OURS:
                    for bar in bars:ax.text(bar.get_x()+bar.get_width()/2,bar.get_height(),'%.3f'%bar.get_height(),ha='center',va='bottom',fontsize=10)
            ax.set_xticks(x);ax.set_xticklabels([d.replace('-day','') for d in days]);ax.set(xlabel='Training days',ylabel=metric+' (°C)',title=ood)
            ax.set_ylim(0,float(np.nanmax(tab.to_numpy()))*1.12);ax.grid(axis='y',alpha=.16)
            ax.legend(fontsize=10,ncol=2)
        save(fig,metric.lower()+'_training_days_grouped_bars')
        OVER.pivot(index=['Train_Days','OOD_Label'],columns='Postprocess_Method',values=metric)[MAIN].to_csv(OUT/(metric.lower()+'_training_days_errors.csv'))


## 2. Building heatmaps

行=建筑，列=预设固定gate。正值表示public的Ours误差更低；每个已有训练规模分别保存。


In [ ]:
if CASES.empty:
    print('本组暂时没有完整配对指标，跳过此类图；稍后重新运行。')
else:
    for metric in METRICS:
        for days in DAYS:
            part=BUILD.loc[BUILD.Train_Days.eq(days)]
            if not len(part):continue
            wide=part.pivot(index=['Building','OOD_Label'],columns='Postprocess_Method',values=metric)
            gain=100*wide[MAIN_RULES].subtract(wide[OURS],axis=0).divide(wide[MAIN_RULES].replace(0,np.nan))
            gain.to_csv(OUT/(metric.lower()+'_'+days+'_building_gains.csv'))
            oods=[o for o in OODS if o in gain.index.get_level_values('OOD_Label')]
            finite=gain.to_numpy(float);finite=finite[np.isfinite(finite)]
            limit=max(1,float(np.max(np.abs(finite)))) if len(finite) else 1
            fig,axes=plt.subplots(1,len(oods),figsize=(6.5*len(oods),5.6),squeeze=False,constrained_layout=True)
            for ax,ood in zip(axes[0],oods):
                grid=gain.xs(ood,level='OOD_Label').reindex([b for b in B_NAMES if b in gain.index.get_level_values('Building')])
                im=ax.imshow(grid,cmap='RdBu',vmin=-limit,vmax=limit,aspect='auto')
                ax.set_xticks(range(6));ax.set_xticklabels(['0','.2','.4','.6','.8','1']);ax.set_xlabel('Fixed gate')
                ax.set_yticks(range(len(grid)));ax.set_yticklabels([B_NAMES.get(b,b) for b in grid.index],fontsize=12);ax.set_title(days+' / '+ood)
                for i in range(len(grid)):
                    for j in range(6):
                        value=grid.iloc[i,j]
                        if np.isfinite(value):ax.text(j,i,'%+.1f%%'%value,ha='center',va='center',color='white' if abs(value)>.65*limit else 'black',fontsize=11)
            fig.colorbar(im,ax=list(axes[0]),label='Ours: error reduction (%)',shrink=.85)
            save(fig,metric.lower()+'_'+days+'_building_heatmaps')


## 3. Forecasting architecture tables

按Architecture×OOD列出Raw、五种固定强度与Ours。每个架构先建筑内平均case，再建筑等权。最小值加粗，不作显著性声明。


In [ ]:
if CASES.empty:
    print('本组暂时没有完整配对指标，跳过此类图；稍后重新运行。')
else:
    for metric in METRICS:
        for days in DAYS:
            part=ARCH.loc[ARCH.Train_Days.eq(days)]
            if not len(part):continue
            tab=part.pivot(index=['Architecture','OOD_Label'],columns='Postprocess_Method',values=metric)[MAIN]
            architectures=[a for a in ['MLP','CNN','GRU','LSTM'] if a in set(part.Architecture)]
            architectures+=sorted(set(part.Architecture)-set(architectures))
            order=pd.MultiIndex.from_product([architectures,OODS],names=tab.index.names)
            tab=tab.reindex(order).dropna(how='all');display(tab.round(4))
            tab.to_csv(OUT/(metric.lower()+'_'+days+'_architecture_table.csv'))
            fig,ax=plt.subplots(figsize=(13,max(4,.4*len(tab)+1.5)));ax.axis('off')
            values=[['%.4f'%v if np.isfinite(v) else '—' for v in row] for row in tab.to_numpy()]
            table=ax.table(cellText=values,colLabels=NAMES,rowLabels=[' / '.join(i) for i in tab.index],loc='center',cellLoc='center')
            table.auto_set_font_size(False);table.set_fontsize(12);table.scale(1,1.6)
            for i,row in enumerate(tab.to_numpy()):
                minimum=np.nanmin(row)
                for j,value in enumerate(row):
                    if np.isfinite(value) and abs(value-minimum)<1e-10:table[(i+1,j)].get_text().set_weight('bold')
            ax.set_title(days+' / '+metric+' (°C)',pad=20);save(fig,metric.lower()+'_'+days+'_architecture_table')


## 4. Fine-grid sensitivity (Full)

沿用0.05完整网格，包含0.75。星号是用本次测试标签回顾性选出的最佳固定gate，仅说明敏感性，不视为无需校准的部署基准。


In [ ]:
if CASES.empty:
    print('本组暂时没有完整配对指标，跳过此类图；稍后重新运行。')
else:
    for metric in METRICS:
        part=OVER.loc[OVER.Train_Days.eq('Full')]
        if not len(part):continue
        table=part.pivot(index='OOD_Label',columns='Postprocess_Method',values=metric)
        oods=[o for o in OODS if o in table.index]
        fig,axes=plt.subplots(1,len(oods),figsize=(6.5*len(oods),4.8),squeeze=False,constrained_layout=True)
        for ax,ood in zip(axes[0],oods):
            row=table.loc[ood];best=row[RULES].idxmin()
            ax.plot(ALPHAS,row[RULES].to_numpy(float),'o-',color='#888888',markersize=4,label='Fixed rule: step 0.05')
            ax.scatter(float(best.split('-')[1]),row[best],marker='*',s=140,color='black',label='Best tested fixed rule',zorder=4)
            ax.axhline(row[OURS],color=COLORS[-1],linestyle='--',linewidth=2,label='Ours: event + phase')
            ax.set(title='Full / '+ood,xlabel='Fixed gate',ylabel=metric+' (°C)');ax.set_xticks(np.arange(0,1.01,.1));ax.tick_params(axis='x',labelsize=11);ax.grid(alpha=.16);ax.legend(fontsize=10)
        save(fig,metric.lower()+'_full_fine_gate_curve')
    print('Outputs:',OUT)
